### *[MIST 4600] Fall 2026 · Week 12*
# **Using LLM APIs: Follow-Along Notebook**

This notebook has the code from today's slides, so you can run it instead of typing it. Run each cell (Shift+Enter) when we reach it in class. The **Your Turn** cells are empty: those you write yourself.

**Before we start:**
1. Set up your Gemini key (slide "Set Up Your Key"): get the key at aistudio.google.com signed in with a **personal Gmail** (AI Studio is turned off for UGA accounts). Then in the **key icon** on the left, add a secret named `GOOGLE_API_KEY` and turn on **Notebook access**.
2. Download `reviews.csv` from eLC (Week 12) and put it in your **MIST 4600** folder in Google Drive, next to this notebook. Then run the setup cell below and allow access to your Drive, just like in the Module 5 lab.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os

# Look for reviews.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "reviews.csv")

if len(found) == 0:
    print("❌ Can't find reviews.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    print("✅ Found reviews.csv in:", FOLDER)

## Part 2 · Your First Call

In [ ]:
from google import genai
from google.colab import userdata

key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=key)

MODEL = "gemini-flash-lite-latest"

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="Suggest a name for a pumpkin spice cold brew. Only the name."
)
print(response.text)

In [ ]:
for i in range(3):
    response = client.models.generate_content(
        model=MODEL,
        contents="Write a fun one-sentence slogan for a fall latte.")
    print(response.text)

## Your Turn 1: Write a Menu Description
Write your code below. `client` and `MODEL` are ready to use.

## Part 3 · An LLM Inside a Function

In [ ]:
import pandas as pd
import time

reviews = pd.read_csv(FOLDER + "reviews.csv")
print(reviews.shape)
reviews.head(3)

In [ ]:
def get_sentiment(review):
    prompt = f"Is this coffee shop review positive, negative, or mixed? Answer with one word only. Review: {review}"
    response = client.models.generate_content(
        model=MODEL, contents=prompt)
    return response.text.strip().lower()

print(get_sentiment("Waited 25 minutes."))

In [ ]:
ALLOWED = ["positive", "negative", "mixed"]

def get_sentiment(review):
    time.sleep(4)       # stay under the limit
    prompt = f"Is this coffee shop review positive, negative, or mixed? Answer with one word only. Review: {review}"
    response = client.models.generate_content(
        model=MODEL, contents=prompt)
    answer = response.text.strip().lower()
    if answer in ALLOWED:
        return answer
    else:
        return "unknown"

This cell sends 30 requests, so it takes about 2 minutes.

In [ ]:
labels = reviews["text"].apply(get_sentiment)
reviews["sentiment"] = labels
reviews["sentiment"].value_counts()

## Your Turn 2: Find the Topic
Write your code below. `reviews` already has the `sentiment` column (from the cell above).

## Part 4 · From Reviews to Action
The cells below use `negative` from Your Turn 2. If you didn't finish it, run this first:

```python
negative = reviews[reviews["sentiment"] == "negative"]
```

In [ ]:
all_text = "\n".join(negative["text"])

prompt = "Here are negative reviews of a coffee shop, one per line. List the 3 most important fixes for the manager, most important first. Use only what the reviews say.\n" + all_text
response = client.models.generate_content(
    model=MODEL, contents=prompt)
print(response.text)

In [ ]:
reviews.sample(5)

In [ ]:
reviews[reviews["stars"] <= 2]

## Your Turn 3: Draft Replies
Write your code below. `negative` is ready to use.